# Track C — Data Quality Check & Scaler Preparation
**Author:** Mohammed Ghannam Alyami  
**Project:** VOLTA — Energy Pressure Forecasting  
**Goal:** Conduct rigorous data quality checks, handle missing values, and prepare leak-free feature scaling on the training split.

In [1]:
# استيراد مكتبات التعامل مع البيانات والتحجيم
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

print('Track C Libraries Imported Successfully!')

Track C Libraries Imported Successfully!


## 1. Load Data & Verify Timeline Completeness
Load the dataset and sort chronologically to maintain time-series integrity.

In [2]:
# 1. قراءة البيانات المدمجة والمنظفة
df = pd.read_csv('../data/processed/volta_clean_hourly.csv')

# 2. ضبط التوقيت الزمني
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values('time').set_index('time')

# 3. عرض ملخص النقص لـ الأهداف الأربعة المعتمدة
print(f'Total Hourly Rows: {len(df)}')
print('Missing values in target columns:')
print(
    df[[
        'price actual',
        'total load actual',
        'renewable_total',
        'fossil_total',
    ]]
    .isnull()
    .sum()
)

Total Hourly Rows: 35064
Missing values in target columns:
price actual         0
total load actual    0
renewable_total      0
fossil_total         0
dtype: int64


## 2. Handle Missing Values
Apply forward filling (`ffill`) for small gaps in time-series data to ensure continuity without using future information.

In [3]:
# 1. تحديد أعمدة الأهداف الرئيسية
features_to_scale = [
    'price actual',
    'total load actual',
    'renewable_total',
    'fossil_total',
]

# 2. أخذ نسخة ومعالجة القيم المفقودة للأعمدة المستهدفة فقط
df_cleaned = df[features_to_scale].copy()
df_cleaned = df_cleaned.ffill().bfill()

# 3. التأكد من اختفاء جميع القيم المفقودة تماماً)
print(
    'Remaining Missing Values in Target Features:',
    df_cleaned.isnull().sum().sum(),
)

Remaining Missing Values in Target Features: 0


## 3. Chronological Train / Test Split
Split the data chronologically into:
- **Train Set:** 2015-01-01 to 2017-12-31
- **Test Set:** 2018-01-01 to 2018-12-31 (Kept strictly untouched)

In [4]:
# التقسيم الزمني المعتمد للسنوات
train_data = df_cleaned.loc[:'2017-12-31']
test_data = df_cleaned.loc['2018-01-01':]

print(f'Train Period: {train_data.index.min()} to {train_data.index.max()}')
print(f'Test Period:  {test_data.index.min()} to {test_data.index.max()}')

Train Period: 2014-12-31 23:00:00+00:00 to 2017-12-31 23:00:00+00:00
Test Period:  2018-01-01 00:00:00+00:00 to 2018-12-31 22:00:00+00:00


## 4. Leak-Free Feature Scaling
Fit the `MinMaxScaler` **strictly on the training data**, then transform both train and test sets independently.

In [5]:

# 1. إنشاء أداة التحجيم
scaler = MinMaxScaler(feature_range=(0, 1))

# 2. ملاءمة الـ Scaler على بيانات التدريب فقط (مهم جداً لمنع Data Leakage)
train_scaled = scaler.fit_transform(train_data[features_to_scale])

# 3. تحويل بيانات الاختبار بناءً على ما تعلمه من التدريب فقط
test_scaled = scaler.transform(test_data[features_to_scale])

# تحويل المخرجات لـ DataFrame لسهولة التسليم لـ Track B
train_scaled_df = pd.DataFrame(
    train_scaled, columns=features_to_scale, index=train_data.index
)
test_scaled_df = pd.DataFrame(
    test_scaled, columns=features_to_scale, index=test_data.index
)

print('Scaling completed without Data Leakage!')
print('Sample Scaled Train Data:')
print(train_scaled_df.head(2))

Scaling completed without Data Leakage!
Sample Scaled Train Data:
                           price actual  total load actual  renewable_total  \
time                                                                          
2014-12-31 23:00:00+00:00      0.521820           0.319666         0.328610   
2015-01-01 00:00:00+00:00      0.517261           0.276008         0.296041   

                           fossil_total  
time                                     
2014-12-31 23:00:00+00:00      0.294467  
2015-01-01 00:00:00+00:00      0.305525  
